In [1]:
# Import dependencies

import os
import numpy as np
import math
from datetime import datetime, date
import matplotlib.pyplot as plt
import rasterio as rio

from functions import *
# from Frameworks import *
import warnings
warnings.filterwarnings("ignore")

# Inputs

### Sample Name

In [2]:
head_dir = "/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/"

# data_time = "Airbus_2024Feb23"
data_time = "Airbus_2024Jan11"
data_time = "Airbus_2024Jun13"

In [3]:
# Directory containing the tif files:
dir_all = os.path.join(head_dir, data_time)
print(f"Processing directory: {dir_all}")


Processing directory: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13


#### Tif images

ALL tif files in the directory

In [4]:
# Get a list of .tif files in the directory:

tif_files, grouped_tif_files = get_file_list(dir_all)

print(f"Found {len(tif_files)} tif files.")

# Example: Print grouped files
for folder, files in grouped_tif_files.items():
    print(f"Folder: {folder}")
    print(f"Files: {files}\n")


Found 441 tif files.
Folder: NovaSAR_01_53540_grd_240411_082440_HH
Files: ['/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53540_grd_240411_082440_HH/NovaSAR_01_53540_grd_13_240411_082440_HH_1/image_HH.tif', '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53540_grd_240411_082440_HH/NovaSAR_01_53540_grd_13_240411_082446_HH_2/image_HH.tif', '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53540_grd_240411_082440_HH/NovaSAR_01_53540_grd_13_240411_082452_HH_3/image_HH.tif', '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53540_grd_240411_082440_HH/NovaSAR_01_53540_grd_13_240411_082457_HH_4/image_HH.tif', '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53540_grd_240411_082440_HH/NovaSAR_01_53540_grd_13_240411_082503_HH_5/image_HH.tif', '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53540_grd_240411_082440_HH/NovaSAR_01_53540_grd_13_240411_082508_HH_6/image_HH.tif', '/mnt/e

# Processing

In [5]:
h= 256  # Half height for patch
w= 256  # Half width for patch

In [20]:
f_ii = 1#5#3#4#2#32#31#30#29#28#27#33#36#35#34#32#21 #20#13#6#5#0#9 # Index of the file to process

In [21]:
im_super_folders = list(grouped_tif_files.keys())
im_super_foldersii = im_super_folders[f_ii]
tif_pathii = grouped_tif_files[ im_super_foldersii ]

print(f"Processing the Images in this folder: {im_super_foldersii}\n")
print(f"tif images directory:")
tif_pathii

Processing the Images in this folder: NovaSAR_01_53556_grd_240412_083034_HH

tif images directory:


['/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53556_grd_240412_083034_HH/NovaSAR_01_53556_grd_13_240412_083034_HH_1/image_HH.tif',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53556_grd_240412_083034_HH/NovaSAR_01_53556_grd_13_240412_083040_HH_2/image_HH.tif',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53556_grd_240412_083034_HH/NovaSAR_01_53556_grd_13_240412_083045_HH_3/image_HH.tif',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53556_grd_240412_083034_HH/NovaSAR_01_53556_grd_13_240412_083051_HH_4/image_HH.tif',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53556_grd_240412_083034_HH/NovaSAR_01_53556_grd_13_240412_083057_HH_5/image_HH.tif',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53556_grd_240412_083034_HH/NovaSAR_01_53556_grd_13_240412_083102_HH_6/image_HH.tif',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53556_g

In [30]:
def ship_patches(im_path, im_name,patch_output_dir, AIS_df, row_AIS, col_AIS, h=64, w=64,uint8=False, plt_ptch=False):
    """
    Extract patches from the image based on AIS data and save them to the specified directory.
    
    Parameters:
    - im_path: Path to the input image file.
    - patch_output_dir: Directory where the patches will be saved.
    - row: List of row indices for AIS points.
    - col: List of column indices for AIS points.
    - h: Half height of the patch.
    - w: Half width of the patch.
    """
    import os
    import rasterio as rio
    from rasterio.windows import Window
    import matplotlib.pyplot as plt
    import shutil
    
    if not os.path.exists(patch_output_dir):
        os.makedirs(patch_output_dir)
    else:
        # If the directory already exists, remove its files and create a new one
        shutil.rmtree(patch_output_dir)
        os.makedirs(patch_output_dir)

    if uint8:
        if not os.path.exists(f"{patch_output_dir}_uint8"):
            os.makedirs(f"{patch_output_dir}_uint8")
        else:
            # If the directory already exists, remove its files and create a new one
            shutil.rmtree(f"{patch_output_dir}_uint8")
            os.makedirs(f"{patch_output_dir}_uint8")

    src = rio.open(im_path)
    im  = src.read(1)

    patch_name_all = []
    ii = 1
    for r_ii, c_ii, sh_t_ii in zip(row_AIS, col_AIS, AIS_df['Ship type']):
        
        if '/' in sh_t_ii:
            sh_t_ii = sh_t_ii.replace('/', '-')
        print(f"sh_t_ii: {sh_t_ii}")
        if r_ii-h < 0 or r_ii+h >= im.shape[0] or c_ii-w < 0 or c_ii+w >= im.shape[1]:
            print(f"Skipping patch for row {r_ii}, col {c_ii} due to out of bounds.")
            patch_name_all.append('')
            continue
        else:
            subii = im[r_ii-h:r_ii+h, c_ii-w:c_ii+w]
            # Adjust metadata:
            out_meta = src.meta.copy()
            out_meta.update({
                "height"   : h*2,
                "width"    : w*2,
                "transform": src.window_transform( Window(c_ii-w, r_ii-h, 2*w, 2*h) ),
                "compress" :'lzw'
            })
            # Write the output:
            patch_nameii = f"{im_name}_patch_{ii}_{sh_t_ii}"
            patch_name_all.append(patch_nameii)
            out_nameii   = f"{patch_output_dir}/{patch_nameii}.tif"
            print(f"out_nameii: {out_nameii}")
            with rio.open(out_nameii, "w", **out_meta) as dest:
                dest.write(subii,1)
            
                subii = subii.astype('float32')
                subii -= subii.mean()
                subii /= 5*subii.std()
                subii += 0.5
                subii = (255 * subii).clip(0, 255).astype('uint8')
                
                out_nameii   = f"{patch_output_dir}_uint8/{patch_nameii}_uint8.tif"
                out_meta.update({
                    "dtype": 'uint8',
                })
                with rio.open(out_nameii, "w", **out_meta) as dest:
                    dest.write(subii,1)

            if plt_ptch:
                plt.figure()
                plt.imshow(subii, cmap='gray', vmin=0, vmax=1500)
                plt.text(h, w, f"{sh_t_ii}", fontsize=10, color='blue', ha='center', va='center')

            ii += 1
    
    # Remove empty directories if they are empty after processing (Because of patches being out of bounds)
    if os.path.isdir(patch_output_dir) and not os.listdir(patch_output_dir):
        os.rmdir(patch_output_dir) 
        os.rmdir(f"{patch_output_dir}_uint8")  

    AIS_df['Patch_name'] = patch_name_all
    
    # Remove rows where column 'Patch_name' is an empty string (out of bounds patches)
    AIS_df = AIS_df[AIS_df['Patch_name'] != '']

    
    return AIS_df

In [31]:
from pathlib import Path


for tii in range( 7,len( tif_pathii ) ):
    # Sample image Full path
    tii_path = Path(tif_pathii[tii][:-12])
    parts = tii_path.parts

    im_name = parts[-1]

    # Find the data acquisition date from the metadata XML file:
    try:
        acqdate0 = f"20{'_'.join(im_name.split('_')[-4:-2])}" # or f"20{tif_files[tii][-31:-18]}" # YYYYMMDD_HHMMSS
        dt       = datetime.strptime(acqdate0, '%Y%m%d_%H%M%S')
        acqdate  = dt.strftime('%d/%m/%Y %H:%M:%S') # DD/MM/YYYY HH:MM:SS
    except:
        acqdate0 = f"20{'_'.join(im_name.split('_')[-3:-1])}" # or f"20{tif_files[tii][-31:-18]}" # YYYYMMDD_HHMMSS
        dt       = datetime.strptime(acqdate0, '%Y%m%d_%H%M%S')
        acqdate  = dt.strftime('%d/%m/%Y %H:%M:%S') # DD/MM/YYYY HH:MM:SS

    print("Raw Data Start Time:", acqdate)

    # AIS data directory:
    ais_name = f"aisdk-{acqdate0[:8][:4]}-{acqdate0[:8][4:6]}-{acqdate0[:8][6:8]}"  # Extract the date part (YYYY-MM-DD)       

    csv_dir = f"{head_dir}AIS_dataset/{ais_name}/{ais_name}.csv" 

    ExtractPatchAndAIS(tii_path=tii_path, AIS_path=csv_dir, h=h, w=w)
    print('')

Raw Data Start Time: 12/04/2024 08:31:13
Sample image (tii) name: NovaSAR_01_53556_grd_13_240412_083113_HH_8


Warning 1: TIFFFetchNormalTag:ASCII value for tag "ImageDescription" contains null byte in value; value incorrectly truncated during reading due to implementation limitations


Skipping patch for row 13608, col -1137 due to out of bounds.


RasterioIOError: Attempt to create new tiff file '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53556_grd_240412_083034_HH/NovaSAR_01_53556_grd_13_240412_083113_HH_8/ship_patches/NovaSAR_01_53556_grd_13_240412_083113_HH_8_patch_3_Towing long/wide.tif' failed: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53556_grd_240412_083034_HH/NovaSAR_01_53556_grd_13_240412_083113_HH_8/ship_patches/NovaSAR_01_53556_grd_13_240412_083113_HH_8_patch_3_Towing long/wide.tif: No such file or directory

## Inshore/Offshore

In [10]:

import geopandas as gpd

# Path to your polyline shapefile
# shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/natural_earth_vector/10m_physical/ne_10m_coastline.shp"  
shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/natural_earth_vector/10m_physical/ne_10m_land.shp"  
# shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/simplified-water-polygons-split-3857/simplified_water_polygons.shp"  
shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/water-polygons-split-4326/water_polygons.shp"  
# shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/land-polygons-complete-4326/land_polygons.shp"
# shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/land-polygons-AOI-4326/land_polygons_AOI.shp"
# Load the shapefile
gdf = gpd.read_file(shapefile_path)
gdf


,x,y,geometry
0,1,41,"POLYGON ((2.00067 40.9995, 0.99933 40.9995, 0...."
1,-11,-72,"POLYGON ((-10.91746 -70.9995, -10.92245 -71.00..."
2,-11,-72,"POLYGON ((-10.75741 -70.9995, -10.73509 -71.00..."
3,148,-11,"POLYGON ((149.00051 -11.0005, 147.99949 -11.00..."
4,-27,-58,"POLYGON ((-25.99907 -56.9995, -25.99907 -58.00..."
...,...,...,...
53294,175,89,"POLYGON ((176.0573 90, 176.0573 88.9995, 174.9..."
53295,176,89,"POLYGON ((177.0573 90, 177.0573 88.9995, 175.9..."
53296,177,89,"POLYGON ((178.0573 90, 178.0573 88.9995, 176.9..."
53297,178,89,"POLYGON ((179.0573 90, 179.0573 88.9995, 177.9..."


In [ ]:
def geodesic_distance_to_polygon(point, polygon):
    import shapely
    from shapely.ops import nearest_points

    # from shapely import LineString, Point, Polygon
    from pyproj import Geod
    """
    Calculate the geodesic distance from a point to a polygon.
    
    Parameters:
    - point: A shapely Point object representing the target point.
    - polygon: A shapely Polygon object representing the polygon.
    
    Returns:
    - distance: The geodesic distance from the point to the polygon boundary.
    """
    # Initialize Geod for geodesic calculations on WGS84 ellipsoid
    geod = Geod(ellps="WGS84")
    
    # dist_deg = shapely.distance(point, polygon) # degrees
    _, p2 = nearest_points(point, polygon.boundary) # Get the nearest point on the polygon boundary to the point
    
    # Calculate the inverse geodesic problem: Returns (azimuth1, azimuth2, distance)
    # _, _, dist_geod = geod.inv( point.x, point.y, point.x + dist_deg/(2**.5), point.y + dist_deg/(2**.5) ) # geodesic distance in meters
    _, _, dist_geod = geod.inv( point.x, point.y, p2.x, p2.y ) # geodesic distance in meters
    
    return dist_geod

In [ ]:
import glob
import os

# Define the base directory
base_dir = f"/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/{data_time}/{im_super_foldersii}/"

# Find all "AIS.csv" files in the directory and subdirectories
AIS_path_all = glob.glob(os.path.join(base_dir, "**", "AIS.csv"), recursive=True)
print(f"Found {len(AIS_path_all)} AIS.csv files.")
AIS_path_all

Found 9 AIS.csv files.


['/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084751_HH_3/AIS.csv',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084756_HH_4/AIS.csv',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084802_HH_5/AIS.csv',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084808_HH_6/AIS.csv',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084813_HH_7/AIS.csv',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084830_HH_10/AIS.csv',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_0

In [ ]:
import pandas as pd
from shapely.geometry import Point, Polygon

dist_thresh = 500  # meters


for AIS_path in AIS_path_all:
    print(f"Processing AIS data from: {AIS_path}")
    ais_data = pd.read_csv(AIS_path)

    ii=0
    for loni, latii in zip(ais_data['Longitude'], ais_data['Latitude']):
        point = Point(loni, latii)
        dist = []
        for polygonii in gdf['geometry'].iloc():
            dist_geod = geodesic_distance_to_polygon(point, polygonii)
            dist.append(dist_geod)
        min_dist = min(dist)
        ais_data.loc[ii,'Dist_to_land'] = min_dist
        if min_dist < dist_thresh:
            ais_data.loc[ii,'Shoreline'] = 'inshore'
        else:
            ais_data.loc[ii,'Shoreline'] = 'offshore'

        # Print the distances for each point
        # print(f"Point ({loni}, {latii}): Distances to land: {min_dist}")
        ii += 1
    ais_data.to_csv(AIS_path, index=False)
    ais_data['Shoreline']


Processing AIS data from: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084751_HH_3/AIS.csv
Processing AIS data from: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084756_HH_4/AIS.csv
Processing AIS data from: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084802_HH_5/AIS.csv
Processing AIS data from: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084808_HH_6/AIS.csv
Processing AIS data from: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_240415_084813_HH_7/AIS.csv
Processing AIS data from: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Jun13/NovaSAR_01_53605_grd_240415_084739_HH/NovaSAR_01_53605_grd_13_2